In [ ]:
%pip install pandas

In [ ]:
import cv2
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib



In [ ]:
def extract_structural_features(image_path):
    # Load in grayscale to focus on structure, not color
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    
    # Resize to a constant square so the "math" is comparable across images
    img = cv2.resize(img, (256, 256)) 

    # 1. Directional Edge Energy (The "Batman Cape" Detector)
    # Sobel X detects vertical lines, Sobel Y detects horizontal lines
    grad_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    
    vertical_energy = np.sum(np.absolute(grad_x))
    horizontal_energy = np.sum(np.absolute(grad_y))
    
    # 2. Projection Variance (The "Stacking" Detector)
    # Portraits usually have high variance vertically (Sky -> Subject -> Ground)
    row_sums = np.sum(img, axis=1) # Vertical profile
    col_sums = np.sum(img, axis=0) # Horizontal profile
    row_variance = np.var(row_sums)
    col_variance = np.var(col_sums)

    return [vertical_energy, horizontal_energy, row_variance, col_variance]

In [ ]:
# 1. Update this path to where your 'landscape' and 'potrait' folders actually live
# If they are in the same folder as your notebook, keep it as "./"
# If they are in a folder called 'data', change it to "./data"
dataset_path = "../images" 

categories = ["landscape", "potrait"] 
X = []
y = []

for label, category in enumerate(categories):
    folder_path = os.path.join(dataset_path, category)
    
    # Debugging: Print exactly where the script is looking
    print(f"Looking for folder at: {os.path.abspath(folder_path)}")
    
    if not os.path.exists(folder_path):
        print(f"⚠️ ERROR: Folder '{category}' not found at {folder_path}")
        print("Try running !ls in a new cell to see your current directory structure.")
        continue
        
    print(f"Processing {category}...")
    
    files = os.listdir(folder_path)
    if len(files) == 0:
        print(f"⚠️ WARNING: Folder '{category}' is empty.")
        
    for img_name in files:
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            path = os.path.join(folder_path, img_name)
            features = extract_structural_features(path)
            if features:
                X.append(features)
                y.append(label)

X = np.array(X)
y = np.array(y)

if len(X) > 0:
    print(f"✅ Success! Loaded {len(X)} images.")
else:
    print("❌ No images were loaded. Please check your dataset_path.")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Step 1: Check if X and y actually have data
if len(X) == 0:
    print("❌ X is empty! No features were extracted. Check your image paths.")
else:
    # Step 2: Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Step 3: Scale the features (Crucial for stability)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Step 4: Train with a simpler set of parameters to avoid over-fitting
    clf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    clf.fit(X_train_scaled, y_train)

    # Step 5: Evaluate
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"✅ Improved Model Accuracy: {acc * 100:.2f}%")
    print(f"Samples analyzed: {len(X)}")
    
    # Save both the model AND the scaler (you'll need the scaler for prediction)
    joblib.dump(clf, "orientation_model.pkl")
    joblib.dump(scaler, "scaler.pkl")

In [ ]:
import pandas as pd
feature_names = ['Vert_Energy', 'Horiz_Energy', 'Row_Var', 'Col_Var']
importances = pd.Series(clf.feature_importances_, index=feature_names)
print("Feature Importance:\n", importances.sort_values(ascending=False))